# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Queue = RF probability (client-holdout model retrained on all rows for scoring) with reason codes from my rule; action depends on position/CTR.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
X = features(df); m = models()['random_forest'].fit(X, df.y)
df['p'] = m.predict_proba(X)[:,1]
_, df['reason'] = baseline_score(df)
pos = pd.to_numeric(df.avg_position, errors='coerce')
df['action'] = np.select([(df.p>=.7)&pos.between(4,20), df.p>=.7, df.p>=.5], ['refresh_and_review_ctr','refresh','review'], 'monitor')
q = df.sort_values('p', ascending=False)
print(q.action.value_counts())
q[['content_id','p','action','reason','impressions_90d','avg_position']].head(15)

action
monitor                   11668
review                    10509
refresh_and_review_ctr     4872
refresh                    2951
Name: count, dtype: int64


,content_id,p,action,reason,impressions_90d,avg_position
7478,content_d5755a0962c8,0.972909,refresh,monitor,5566,24.1
2063,content_ce8dacc2467b,0.970529,refresh,monitor,9710,26.4
15272,content_12c48bcc972c,0.970183,refresh,monitor,10171,28.0
29353,content_a28fe6cce32e,0.969711,refresh,monitor,4282,25.5
21344,content_af3e1659ba29,0.966590,refresh,monitor,3288,26.8
5998,content_e931ef5c808e,0.961004,refresh,monitor,3750,31.3
6830,content_732df0136343,0.960384,refresh_and_review_ctr,striking_distance,3731,18.5
28381,content_73bd82df022f,0.960335,refresh,monitor,3289,25.6
18891,content_54f92ea426e9,0.960170,refresh,monitor,8156,22.0
12809,content_57f3561919ff,0.958971,refresh,monitor,4434,23.7


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Users: content editors triaging existing pages weekly. Valid for this portfolio's page types; not valid for new pages (no history), pages with <100 impressions (noisy label), or as a forecast.

In [2]:
print('pages below 100 impressions (exclude from action):', int((df.impressions_90d<100).sum()))

pages below 100 impressions (exclude from action): 7994


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human must check the live page, intent, seasonality, and client priorities before editing. Never automate: publishing/rewrites, deletions or noindex decisions.

In [3]:
print('high-prob pages needing human check:', int((df.p>=.7).sum()))

high-prob pages needing human check: 7823


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain triggers: weekly P@50 of last queue vs realised trend drops below baseline; base rate shifts >5pts; new content types appear.

In [4]:
print('current base rate:', round(df.y.mean(),3))

current base rate: 0.542


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports to work/outputs/ (CSV is gitignored; summary JSON committed).

In [5]:
q[['content_id','p','action','reason']].to_csv(OUT/'action_queue.csv', index=False)
import json; json.dump({'action_counts': q.action.value_counts().to_dict()}, open(OUT/'action_summary.json','w'), indent=1); print('written')

written


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.